# Deposit Attrition EDA — v10b · corrections to the v10 run

**Payment Knowledge Graph (PKG) · PNC Treasury Management · Data Science**

Three defects in v10, one of them mine and consequential, plus the answer to *why did
`payment_only` beat the combined sets on dollars*.

---

### 1 · The isotonic fix made calibration worse, not better

| set | version | calib error | top-decile ratio |
|---|---|---|---|
| deposit_only | raw | 0.516 | 1.95 |
| deposit_only | **v10 isotonic** | **1.590** | **0.435** |
| all_features | raw | 0.495 | 1.73 |
| all_features | **v10 isotonic** | **1.389** | **0.470** |

It went from under-predicting ~2× to **over-predicting ~2.2×** and tripled the error. The cause is
specific: the calibration slice is taken from `TR`, which is **case-control sampled at 15%**. Its
prevalence is roughly 5× population, so fitting an unweighted isotonic map on it **undoes the
King & Zeng prior correction the intercept had just applied**. The arithmetic matches — top-decile
prediction moved 0.083 → 0.381, a factor of 4.6, against a sampling ratio of ~5.2.

Compounding it: `pav()` accepts a `w` argument and the binned rewrite **silently ignored it**.
Fixed in §1, and the calibration slice is now weighted back to population prevalence.

Isotonic is monotone, so **the §7 ordering did not change** — but every probability and every
expected-dollar figure in v10 is inflated by roughly 2.2×.

---

### 2 · Why `payment_only` "won" on dollars — it is an α artefact, not a capability

The queue scores `p × balance^0.75`. From v9 §9, **base rate falls with client size**: 6.1% in
balance decile 1 against 1.8% in decile 10. Large clients churn *less*.

`deposit_only`, `both` and `all_features` all contain `ld_bl_log_bal`, so they **learn that**, and
their `p` carries a negative size coefficient which partly cancels the `balance^0.75` weight.
`payment_only` has no balance feature at all, is blind to size, and the value weight drags the
largest balances to the top unopposed.

The v10 numbers fit exactly. Reconstructing reach from the 1% column:

| set | dollars reached | conversations | break-even |
|---|---|---|---|
| payment_only | **$566m** | ~1,757 | **0.078%** |
| all_features | $491m | ~2,644 | 0.135% |
| deposit_only | $193m | ~5,700 | 0.220% |

It reaches more money through **fewer distinct clients** — repeatedly flagging the same whales —
while its AUC is *worse* everywhere (0.7486 against 0.7976 at H=6). **It wins by not knowing
something true.** §5 measures the size-blindness directly and §6 re-runs the comparison at
**α = 1 with a calibrated probability**, where `p × balance` *is* expected dollars and size cannot
be double-counted.

Two further config problems fed this: §7 was hardcoded to `ALPHA_BASE = 0.75` while §9 found
**α = 0.5** best for `all_features` ($535.7m against $490.7m), and isotonic **pooling creates
ties** whose ranking collapses to pure balance — with the degree of pooling differing by set.

---

### 3 · `D_money_move` — the premise partly fails, and the failure is informative

Median gap is **2 months**, not the 6–8 the decay curve implied (p25 0, p75 6).

Both are right. The decay curve is **dollar-weighted**; `D` is a **client-count median**. The
reconciliation is that *large clients move early and move big, small clients drain at the end* —
the same concentration story from a new angle. §7 splits the gap by balance decile, because if the
top deciles show a 6–9 month gap then `D` is still the right target and belongs on a
value-weighted definition.

---

### Also in this notebook
- Permutation control across **all seven folds** — `both` returned 0.4266 on a single fold in v10
  and was flagged, and one fold is not enough to clear or condemn it.
- **`B_bal_exit` folded into the pool and the savings engine.** $7.35bn across 2,633 clients, mean
  $2.8m against A's $858k. Excluding it understates the addressable pool by **72%**.
- Marginal return computed on the **efficient frontier** rather than raw successive differences,
  which produced −$156,678 at k=100 in v10.
- The lead ladder normalised **per client**, since the 12-month rung runs over fewer clients than
  the 9-month rung and is not comparable as a raw sum.

In [ ]:
# =====================================================================
# 0 · SETUP — reuses the v10 kernel where possible
# =====================================================================
import warnings, time, math, numpy as np, pandas as pd
from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F
from pyspark import StorageLevel
from IPython.display import display, HTML
from pathlib import Path
warnings.filterwarnings("ignore")

if "spark" not in dir():
    spark = (SparkSession.builder.appName("pkg_attrition_eda_v10b")
             .config("spark.sql.shuffle.partitions", "800")
             .config("spark.sql.execution.arrow.pyspark.enabled", "false")
             .enableHiveSupport().getOrCreate())
if "OUT_DIR" not in dir():
    HDFS_V2 = "hdfs://nameservice1/user/pk36814/attrition_v2"
    HDFS_V6 = "hdfs://nameservice1/user/pk36814/attrition_v6"
    HDFS_V9 = "hdfs://nameservice1/user/pk36814/attrition_v9"
    HDFS_DIR = "hdfs://nameservice1/user/pk36814/attrition_v10"
    OUT_DIR = Path("/projects/DSI/sa15474/repos/pkg/eda/attrition_v10")
    def v2(n): return f"{HDFS_V2.rstrip('/')}/{n}"
    def v6(n): return f"{HDFS_V6.rstrip('/')}/{n}"
    def v9(n): return f"{HDFS_V9.rstrip('/')}/{n}"
    def hp(n): return f"{HDFS_DIR.rstrip('/')}/{n}"
    NEG_SAMPLE, L2, MIN_TRAIN_POS = 0.15, 2.0, 200
    PRIMARY_H, PRIMARY_DEF, SEED = 6, "A_full_exit", 20260909
    HORIZON_GRID = [1, 3, 6, 9, 12]
    CAPACITY = [50, 100, 250, 500, 1000, 2500, 5000]
    QUEUE_K, RM_COST_PER_CALL, ANNUALISE = 1000, 250.0, 12.0/7.0
    P_SAVE_GRID, P_SAVE_BASE = [0.01, 0.05, 0.10, 0.20, 0.50], 0.10
    LEAD_GRID, BAL_FLOOR, MAX_ROWS = [1, 2, 3, 4, 6, 9, 12], 1000.0, 60
    MOVE_ATRISK_FRAC, JACKKNIFE_TOP = 0.50, [0, 10, 50, 100]
OUT_DIR.mkdir(parents=True, exist_ok=True)
ALPHA_GRID = [0.0, 0.25, 0.5, 0.75, 1.0]
CAL_MONTHS = 2

HAVE_MPL = True
try:
    import matplotlib; matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    from matplotlib.ticker import FuncFormatter
    plt.rcParams.update({"figure.dpi": 120, "font.size": 9,
        "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
        "grid.color": "#E5E7EB", "grid.linewidth": .7, "axes.edgecolor": "#9AA1AC",
        "figure.facecolor": "white", "axes.facecolor": "white"})
except Exception:
    HAVE_MPL = False
ACC, ACC2, GOOD, WARN, GREY, INK = "#C1440E", "#4A6FA5", "#2F6F4E", "#B8860B", "#9AA1AC", "#16181D"
FSCOL = {"deposit_only": GREY, "payment_only": ACC2, "both": GOOD, "all_features": ACC}
FS_ORDER = ["deposit_only", "payment_only", "both", "all_features"]

def usd(v):
    if v is None or (isinstance(v, float) and not np.isfinite(v)): return "—"
    a = abs(v)
    if a >= 1e9: return f"${v/1e9:,.2f}bn"
    if a >= 1e6: return f"${v/1e6:,.1f}m"
    if a >= 1e3: return f"${v/1e3:,.0f}k"
    return f"${v:,.0f}"
def _dec(s):
    o = s
    for c, t in s.dtypes:
        if t.startswith("decimal"): o = o.withColumn(c, F.col(c).cast("double"))
    return o
def disp(o, title=None, n=None, save=None):
    n = MAX_ROWS if n is None else n
    out = _dec(o).limit(n).toPandas() if hasattr(o, "toPandas") else (
        o.copy() if isinstance(o, pd.DataFrame) else pd.DataFrame(o))
    if save: out.to_csv(OUT_DIR / f"{save}.csv", index=False)
    if title: display(HTML(f"<div style='font:600 13px/1.6 IBM Plex Sans,sans-serif;"
                           f"margin:10px 0 2px;color:#111'>{title}</div>"))
    display(out); return out
def kv(pairs, title=None, save=None):
    items = list(pairs); labs = [k for k, _ in items]
    d = sorted({k for k in labs if labs.count(k) > 1})
    if d: raise ValueError(f"kv(): duplicate labels {d}")
    return disp(pd.DataFrame({"metric": labs, "value": [v for _, v in items]}),
                title=title, n=len(items), save=save)
def collect_pd(sdf, label=""):
    n = sdf.count(); t0 = time.time(); out = _dec(sdf).toPandas()
    print(f"  collected {label}: {n:,} x {out.shape[1]} in {time.time()-t0:,.0f}s"); return out
def _fin(fig, title, sub=None, save=None):
    if sub: fig.text(0.005, 0.965, sub, fontsize=8, color="#6B7280", va="top")
    fig.suptitle(title, fontsize=11, fontweight="bold", x=0.005, ha="left", y=1.0)
    fig.tight_layout(rect=[0, 0, 1, 0.94 if sub else 0.96])
    if save: fig.savefig(OUT_DIR / f"{save}.png", bbox_inches="tight")
    display(fig); plt.close(fig)
def bar_grouped(df, x, series, title, sub=None, ylab="", fmt=usd, save=None,
                colors=None, figsize=(9.5, 4.2)):
    if not HAVE_MPL: return disp(df.reset_index(), title=title)
    fig, ax = plt.subplots(figsize=figsize); n = len(series); w = 0.8/n
    idx = np.arange(len(df))
    for i, s in enumerate(series):
        b = ax.bar(idx+(i-(n-1)/2)*w, df[s].values, w, label=s,
                   color=(colors or {}).get(s))
        for r, v in zip(b, df[s].values):
            if np.isfinite(v): ax.text(r.get_x()+r.get_width()/2, v, fmt(v),
                                       ha="center", va="bottom", fontsize=7)
    ax.set_xticks(idx); ax.set_xticklabels([str(i) for i in df.index])
    ax.set_xlabel(x); ax.set_ylabel(ylab); ax.margins(y=.18)
    ax.legend(frameon=False, fontsize=8, ncol=min(n, 4)); _fin(fig, title, sub, save)
def heat(df, title, sub=None, fmt=usd, save=None, xlab="", ylab="", figsize=(9.5, 4.2)):
    if not HAVE_MPL: return disp(df.reset_index(), title=title)
    fig, ax = plt.subplots(figsize=figsize); V = df.values.astype(float)
    im = ax.imshow(V, cmap="OrRd", aspect="auto")
    ax.set_xticks(range(df.shape[1])); ax.set_xticklabels(df.columns, fontsize=8)
    ax.set_yticks(range(df.shape[0])); ax.set_yticklabels(df.index, fontsize=8)
    ax.set_xlabel(xlab); ax.set_ylabel(ylab); ax.grid(False)
    mx = np.nanmax(V)
    for i in range(df.shape[0]):
        for j in range(df.shape[1]):
            if np.isfinite(V[i, j]):
                ax.text(j, i, fmt(V[i, j]), ha="center", va="center", fontsize=7.5,
                        color=("white" if V[i, j] > .62*mx else INK))
    fig.colorbar(im, ax=ax, shrink=.85, pad=.015); _fin(fig, title, sub, save)
def lines(df, title, sub=None, ylab="", xlab="", fmt=None, save=None, colors=None,
          logx=False, figsize=(9.5, 4.2)):
    if not HAVE_MPL: return disp(df.reset_index(), title=title)
    fig, ax = plt.subplots(figsize=figsize)
    for c in df.columns:
        ax.plot(df.index, df[c].values, marker="o", ms=4.5, lw=2,
                color=(colors or {}).get(c), label=str(c))
    if logx:
        ax.set_xscale("log"); ax.set_xticks(list(df.index))
        ax.set_xticklabels([f"{i:,}" for i in df.index], fontsize=8)
    if fmt: ax.yaxis.set_major_formatter(FuncFormatter(lambda v, p: fmt(v)))
    ax.set_xlabel(xlab); ax.set_ylabel(ylab)
    ax.legend(frameon=False, fontsize=8, ncol=min(len(df.columns), 4))
    _fin(fig, title, sub, save)

def auc(y, s):
    y = np.asarray(y, float); s = np.asarray(s, float)
    ok = np.isfinite(s) & np.isfinite(y); y, s = y[ok], s[ok]
    n1 = float(y.sum()); n0 = float(len(y)-n1)
    if n1 == 0 or n0 == 0: return np.nan
    r = pd.Series(s).rank(method="average").to_numpy()
    return float((r[y == 1].sum()-n1*(n1+1)/2.0)/(n1*n0))
def logit_irls(X, y, l2=L2, mi=60, tol=1e-9):
    X = np.asarray(X, np.float64); y = np.asarray(y, np.float64)
    b = np.zeros(X.shape[1]); R = l2*np.eye(X.shape[1]); R[0, 0] = 0.0
    for _ in range(mi):
        eta = np.clip(X@b, -30, 30); mu = 1/(1+np.exp(-eta))
        w = np.maximum(mu*(1-mu), 1e-6); z = eta+(y-mu)/w; XtW = X.T*w
        try: bn = np.linalg.solve(XtW@X+R, XtW@z)
        except np.linalg.LinAlgError:
            bn = np.linalg.lstsq(XtW@X+R, XtW@z, rcond=None)[0]
        if np.max(np.abs(bn-b)) < tol: b = bn; break
        b = bn
    return b
def fit_spec(tr, cols, l2=L2, s=NEG_SAMPLE):
    X = tr[cols].to_numpy(np.float64); y = tr["y"].to_numpy(np.float64)
    keep = X.std(axis=0) > 1e-9
    ck = [c for c, k in zip(cols, keep) if k]
    if not ck or y.sum() < 2: return None
    Xk = X[:, keep]; mu = Xk.mean(0); sd = Xk.std(0)
    b = logit_irls(np.column_stack([np.ones(len(Xk)), (Xk-mu)/sd]), y, l2)
    return dict(cols=ck, beta=b[1:]/sd, raw=b,
                b0=float(b[0]-float(np.sum(b[1:]*mu/sd))) + math.log(s))
def predict_p(sp, df):
    if sp is None: return np.full(len(df), np.nan)
    eta = sp["b0"] + df[sp["cols"]].to_numpy(np.float64) @ sp["beta"]
    return 1.0/(1.0+np.exp(-np.clip(eta, -30, 30)))
print("helpers ready")


In [ ]:
# =====================================================================
# 1 · THE FIXED ISOTONIC                                 [OUTPUT BLOCK 1]
# =====================================================================
def pav(x, y, w=None, nbins=200):
    """Weighted pool-adjacent-violators on quantile bins.

    TWO FIXES over v10:
      1. `w` was accepted and then SILENTLY IGNORED by the binned rewrite.
         It is now carried through the binning and the merge.
      2. The caller must pass population weights. The calibration slice comes
         from a case-control TRAINING sample at NEG_SAMPLE, so its prevalence
         is ~1/NEG_SAMPLE times population. Fitting an unweighted map on it
         UNDOES the King & Zeng prior correction the intercept just applied —
         which is exactly what happened in v10 (top-decile ratio 1.95 -> 0.44).

    Binning first is what makes this usable: a raw PAV deleting from a python
    list is O(n^2) and does not return on a 150k-row slice."""
    x = np.asarray(x, float); y = np.asarray(y, float)
    w = np.ones_like(y) if w is None else np.asarray(w, float)
    ok = np.isfinite(x) & np.isfinite(y) & np.isfinite(w) & (w > 0)
    x, y, w = x[ok], y[ok], w[ok]
    if len(x) == 0: return np.array([0.0]), np.array([0.0])
    nb = int(min(nbins, max(2, len(x)//50)))
    edges = np.unique(np.quantile(x, np.linspace(0, 1, nb+1)))
    if len(edges) < 3:
        return (np.array([float(x.min()), float(x.max())]),
                np.array([float(np.average(y, weights=w))]*2))
    idx = np.clip(np.searchsorted(edges, x, side="right")-1, 0, len(edges)-2)
    d = pd.DataFrame({"b": idx, "yw": y*w, "xw": x*w, "w": w})
    g = d.groupby("b").agg(yw=("yw", "sum"), xw=("xw", "sum"), w=("w", "sum"))
    g["x"] = g.xw/g.w; g["y"] = g.yw/g.w
    g = g.sort_values("x")
    sx, sy, sw = [], [], []
    for xi, yi, wi in zip(g.x.values, g.y.values, g.w.values):
        sx.append(float(xi)); sy.append(float(yi)); sw.append(float(wi))
        while len(sy) > 1 and sy[-2] > sy[-1]:
            nw = sw[-2]+sw[-1]
            sy[-2] = (sy[-2]*sw[-2] + sy[-1]*sw[-1])/nw
            sw[-2] = nw; sx[-2] = sx[-1]
            sy.pop(); sw.pop(); sx.pop()
    return np.asarray(sx), np.asarray(sy)

def apply_iso(knots, p):
    kx, ky = knots; p = np.asarray(p, float)
    if len(ky) == 0 or not np.isfinite(ky).any() or np.nanmax(ky) <= 0: return p
    return np.interp(p, kx, ky, left=ky[0], right=ky[-1])

def cal_weights(y):
    """Positives kept whole, negatives sampled at NEG_SAMPLE -> weight 1/NEG_SAMPLE
    to restore population prevalence before fitting the isotonic map."""
    return np.where(np.asarray(y) == 1, 1.0, 1.0/NEG_SAMPLE)

# quick self-check on synthetic data: the weighted map must recover the
# POPULATION rate, the unweighted one must recover the SAMPLED rate
_rng = np.random.default_rng(1)
_n = 60000
_p = _rng.beta(1.2, 25, _n)                 # population probabilities
_y = (_rng.random(_n) < _p).astype(float)
_keep = (_y == 1) | (_rng.random(_n) < NEG_SAMPLE)
_xs, _ys = _p[_keep], _y[_keep]
_un = apply_iso(pav(_xs, _ys), _p).mean()
_wt = apply_iso(pav(_xs, _ys, cal_weights(_ys)), _p).mean()
kv([("true population rate", f"{_y.mean():.4%}"),
    ("rate in the case-control slice", f"{_ys.mean():.4%}"),
    ("mean prediction, UNWEIGHTED isotonic (the v10 bug)", f"{_un:.4%}"),
    ("mean prediction, WEIGHTED isotonic (fixed)", f"{_wt:.4%}"),
    ("inflation factor removed", f"{_un/max(_wt, 1e-9):.2f}x")],
   title="1a &middot; <b>Self-check on synthetic data.</b> The unweighted map reproduces the "
         "sampled prevalence; the weighted map reproduces the population prevalence. That "
         "difference is the whole v10 calibration defect", save="v10b_pav_check")
assert abs(_wt - _y.mean()) < 0.2*_y.mean(), "weighted isotonic failed its own self-check"


In [ ]:
# =====================================================================
# 2 · REFIT WITH CORRECTED CALIBRATION                   [OUTPUT BLOCK 2]
# =====================================================================
# Reuses the v10 kernel's collected frames when they are present; rebuilds
# them from the persisted risk set otherwise.
if "TRp" not in dir() or "TEp" not in dir():
    print("  rebuilding TRp / TEp from attrition_v9/risk_set_v9 …")
    cust_month = spark.read.parquet(v2("panel_customer_month"))
    YMMAP = cust_month.select("ym", "m_idx").distinct()
    M_MIN, M_MAX = [int(x) for x in cust_month.agg(F.min("m_idx"), F.max("m_idx")).collect()[0]]
    lab = spark.read.parquet(v6("labels_customer"))
    BAL = spark.read.parquet(v9("balance"))
    RISK = spark.read.parquet(v9("risk_set_v9"))
    ALLC = RISK.columns
    NEWP = ("cptyn_", "cptya_", "fin_out2", "fin_in", "tim_", "railmix", "conc_",
            "selfpay", "acc_", "bl_")
    def _pair(ld): return sorted(set(ld+[f"md_{c[3:]}" for c in ld if f"md_{c[3:]}" in ALLC]))
    V7_LD = sorted([c for c in ALLC if c.startswith("ld_")
                    and not any(c.startswith("ld_"+p) for p in NEWP)])
    DEP_CORE = [c for c in V7_LD if c == "ld_bal_live"]
    PAY_CORE = [c for c in V7_LD if c not in DEP_CORE]
    def blk(*pfx): return _pair(sorted([c for c in ALLC
                                        if any(c.startswith("ld_"+p) for p in pfx)]))
    DEPOSIT = _pair(DEP_CORE)+blk("bl_")+blk("acc_")
    PAYMENT = _pair(PAY_CORE); BOTH = sorted(set(DEPOSIT+PAYMENT))
    EXTRA = blk("fin_in")+blk("cptya_out")+blk("fin_out2")+ \
            sorted([c for c in ALLC if "rec_" in c and c.startswith(("ld_", "md_"))])
    ALLF = sorted(set(BOTH+EXTRA))
    FSETS = {"deposit_only": DEPOSIT, "payment_only": PAYMENT, "both": BOTH,
             "all_features": ALLF}
    ORIGINS = [t for t in range(M_MIN+18, M_MAX-min(HORIZON_GRID)+1) if t <= M_MAX-PRIMARY_H]
    DL = spark.read.parquet(hp("labels_D")) if True else None
    NEED = sorted(set(["cust_pwr_id", "m_idx", "event_A", "event_B",
                       "bar_now", "bar_med12"]+ALLF) & set(ALLC))
    RS = (RISK.select(*NEED)
          .join(DL.withColumnRenamed("q_D_money_move", "event_D"), "cust_pwr_id", "left")
          .join(BAL.select("cust_pwr_id", "m_idx", "bal_med12", "bal_now"),
                ["cust_pwr_id", "m_idx"], "left")
          .withColumn("d_atrisk", ((F.coalesce("bal_med12", F.lit(0.0)) <= BAL_FLOOR) |
                      (F.col("bal_now") >= MOVE_ATRISK_FRAC*F.col("bal_med12"))).cast("int"))
          .drop("bal_med12", "bal_now"))
    is_pos = (F.col("event_A").between(F.col("m_idx")+1, F.col("m_idx")+max(HORIZON_GRID)) |
              F.col("event_D").between(F.col("m_idx")+1, F.col("m_idx")+max(HORIZON_GRID)) |
              F.col("event_B").between(F.col("m_idx")+1, F.col("m_idx")+max(HORIZON_GRID)))
    TR_RAW = collect_pd(RS.filter(F.col("m_idx") <= max(ORIGINS)-min(HORIZON_GRID))
                        .withColumn("_u", (F.abs(F.hash(F.concat_ws("|", "cust_pwr_id",
                                    F.col("m_idx").cast("string"), F.lit(SEED))))%100000)/100000.)
                        .filter(is_pos | (F.col("_u") < NEG_SAMPLE)), "TRAIN")
    TE_RAW = {t: collect_pd(RS.filter(F.col("m_idx") == t), f"TEST {t}") for t in ORIGINS}
    def prep(d):
        d = d.copy()
        for c in d.columns:
            if c.startswith("ld_"): d[c] = pd.to_numeric(d[c], errors="coerce").fillna(0.0)
            elif c.startswith("md_"): d[c] = pd.to_numeric(d[c], errors="coerce").fillna(1.0)
        for c in ["bar_now", "bar_med12"]:
            d[c] = pd.to_numeric(d[c], errors="coerce").fillna(0.0).clip(lower=0)
        d["d_atrisk"] = pd.to_numeric(d.get("d_atrisk", 1), errors="coerce").fillna(1).astype(int)
        return d
    TRp = prep(TR_RAW); TEp = {k: prep(v) for k, v in TE_RAW.items()}
else:
    print("  reusing TRp / TEp from the v10 kernel")

EVCOL = {"A_full_exit": "event_A", "D_money_move": "event_D", "B_bal_exit": "event_B"}
def label(d, defn, H):
    """AB_any is the union — the money leaves whether the account closes or is
    simply drained, and v10 counted only the first."""
    if defn == "AB_any":
        ev = pd.concat([pd.to_numeric(d["event_A"], errors="coerce"),
                        pd.to_numeric(d["event_B"], errors="coerce")], axis=1).min(axis=1)
    else:
        ev = pd.to_numeric(d[EVCOL[defn]], errors="coerce")
    t = pd.to_numeric(d["m_idx"], errors="coerce")
    y = ((ev > t) & (ev <= t+H)).astype(float)
    keep = ((t+H <= M_MAX) | (y == 1)) & (ev.isna() | (ev > t))
    if defn == "D_money_move": keep = keep & (d["d_atrisk"] == 1)
    o = d.loc[keep].copy(); o["y"] = y.loc[keep].values; o["event_m"] = ev.loc[keep].values
    return o

def run(defn, horizons=(PRIMARY_H,)):
    folds, scored = [], []
    for H in horizons:
        for T in ORIGINS:
            if T+H > M_MAX: continue
            trf = label(TRp[TRp.m_idx <= T-H], defn, H)
            if len(trf) == 0 or trf.y.sum() < MIN_TRAIN_POS: continue
            cut = trf.m_idx.max()-CAL_MONTHS
            fit_df, cal_df = trf[trf.m_idx <= cut], trf[trf.m_idx > cut]
            if len(cal_df) < 5000 or cal_df.y.sum() < 20: fit_df = cal_df = trf
            te = label(TEp[T], defn, H)
            if len(te) == 0 or te.y.sum() < 1: continue
            out = te[["cust_pwr_id", "m_idx", "y", "event_m", "bar_now", "bar_med12"]].copy()
            out["H"], out["defn"] = H, defn
            wc = cal_weights(cal_df.y.values)
            for fs, cols in FSETS.items():
                cols = [c for c in cols if c in fit_df.columns]
                sp = fit_spec(fit_df, cols)
                praw = predict_p(sp, te)
                out[f"praw_{fs}"] = praw
                out[f"pold_{fs}"] = apply_iso(pav(predict_p(sp, cal_df),
                                                  cal_df.y.values), praw)      # v10, unweighted
                out[f"p_{fs}"] = apply_iso(pav(predict_p(sp, cal_df),
                                               cal_df.y.values, wc), praw)     # fixed
                folds.append(dict(defn=defn, H=H, origin=T, feature_set=fs,
                                  auc=auc(te.y.values, praw), base=float(te.y.mean()),
                                  n_feat=len(sp["cols"]) if sp else 0))
            scored.append(out)
    return pd.DataFrame(folds), pd.concat(scored, ignore_index=True)

t0 = time.time(); SC, FD = {}, {}
for d in ["A_full_exit", "AB_any"]:
    FD[d], SC[d] = run(d)
    print(f"  {d}: {len(SC[d]):,} scored rows ({time.time()-t0:,.0f}s)")
S = SC[PRIMARY_DEF]


In [ ]:
# =====================================================================
# 3 · CALIBRATION — raw vs v10 isotonic vs fixed          [OUTPUT BLOCK 3]
# =====================================================================
rows = []
for fs in FS_ORDER:
    for tag, col in [("raw", f"praw_{fs}"), ("v10 isotonic (unweighted)", f"pold_{fs}"),
                     ("v10b isotonic (weighted)", f"p_{fs}")]:
        d = S[["y", col]].dropna().rename(columns={col: "p"})
        d["dec"] = pd.qcut(d.p.rank(method="first"), 10, labels=False)+1
        g = d.groupby("dec", as_index=False).agg(pred=("p", "mean"), real=("y", "mean"))
        top = g[g.dec == 10].iloc[0]
        rows.append(dict(feature_set=fs, version=tag,
                         calib_error=float(np.abs(g.real-g.pred).sum()/g.real.sum()),
                         mean_pred=float(d.p.mean()), mean_real=float(d.y.mean()),
                         top_pred=float(top.pred), top_real=float(top.real),
                         top_ratio=float(top.real/top.pred)))
CAL = pd.DataFrame(rows)
disp(CAL.round(4), title="3a &middot; <b>Calibration, three ways.</b> "
     "<code>top_ratio</code> near 1.0 is the target — that decile is the queue. v10 pushed it "
     "from 1.95 (under) to 0.44 (over); the weighted map should land near 1", n=20,
     save="v10b_calibration")
P = CAL.pivot_table(index="feature_set", columns="version", values="top_ratio").reindex(FS_ORDER)
bar_grouped(P, "feature set", list(P.columns),
            "3a · Top-decile calibration ratio — 1.00 is correct",
            "realised rate / predicted rate in the decile the queue works from",
            ylab="ratio", fmt=lambda v: f"{v:.2f}", save="v10b_calib_ratio")
if HAVE_MPL:
    fig, axes = plt.subplots(1, 3, figsize=(12.5, 3.9), sharey=True)
    for ax, (tag, pre) in zip(axes, [("raw", "praw_"), ("v10 unweighted", "pold_"),
                                     ("v10b weighted", "p_")]):
        for fs in FS_ORDER:
            d = S[["y", pre+fs]].dropna().rename(columns={pre+fs: "p"})
            d["dec"] = pd.qcut(d.p.rank(method="first"), 10, labels=False)+1
            g = d.groupby("dec").agg(pred=("p", "mean"), real=("y", "mean"))
            ax.plot(g.pred, g.real, marker="o", ms=4, lw=1.6, color=FSCOL[fs], label=fs)
        lim = max(ax.get_xlim()[1], ax.get_ylim()[1])
        ax.plot([0, lim], [0, lim], ls=":", color=INK, lw=1)
        ax.set_title(tag, fontsize=9.5); ax.set_xlabel("predicted")
    axes[0].set_ylabel("realised"); axes[0].legend(frameon=False, fontsize=7.5)
    _fin(fig, "3b · Reliability curves — the middle panel is the v10 defect",
         "points above the diagonal are under-predicted, below are over-predicted",
         save="v10b_reliability")
_fix = CAL[CAL.version.str.startswith("v10b")]
kv([("worst top-decile ratio, raw",
     f"{CAL[CAL.version=='raw'].top_ratio.max():.2f}"),
    ("worst top-decile ratio, v10 isotonic",
     f"{CAL[CAL.version.str.startswith('v10 ')].top_ratio.min():.2f}"),
    ("top-decile ratio range, v10b weighted",
     f"{_fix.top_ratio.min():.2f} – {_fix.top_ratio.max():.2f}"),
    ("mean calibration error, v10 isotonic",
     f"{CAL[CAL.version.str.startswith('v10 ')].calib_error.mean():.3f}"),
    ("mean calibration error, v10b weighted", f"{_fix.calib_error.mean():.3f}"),
    ("every v10 expected-dollar figure was inflated by roughly",
     f"{1/max(CAL[CAL.version.str.startswith('v10 ')].top_ratio.mean(), 1e-9):.1f}x")],
   title="3c &middot; Verdict", save="v10b_calib_verdict")


In [ ]:
# =====================================================================
# 4 · PERMUTATION CONTROL — ALL FOLDS                    [OUTPUT BLOCK 4]
# =====================================================================
# v10 ran one fold; `both` came back 0.4266 and was flagged. One fold cannot
# clear or condemn it, so this runs every fold and reports the distribution.
rng = np.random.default_rng(SEED)
rows = []
for T in ORIGINS:
    trf = label(TRp[TRp.m_idx <= T-PRIMARY_H], PRIMARY_DEF, PRIMARY_H)
    te = label(TEp[T], PRIMARY_DEF, PRIMARY_H)
    if len(trf) == 0 or trf.y.sum() < MIN_TRAIN_POS or te.y.sum() < 1: continue
    sh = trf.copy(); sh["y"] = rng.permutation(sh.y.values)
    for fs, cols in FSETS.items():
        cols = [c for c in cols if c in trf.columns]
        rows.append(dict(origin=T, feature_set=fs,
                         auc_real=auc(te.y.values, predict_p(fit_spec(trf, cols), te)),
                         auc_perm=auc(te.y.values, predict_p(fit_spec(sh, cols), te))))
PERM = pd.DataFrame(rows)
PS = (PERM.groupby("feature_set", as_index=False)
      .agg(folds=("auc_perm", "size"), real_mean=("auc_real", "mean"),
           perm_mean=("auc_perm", "mean"), perm_sd=("auc_perm", "std"),
           perm_min=("auc_perm", "min"), perm_max=("auc_perm", "max"))).set_index("feature_set")
PS = PS.reindex(FS_ORDER)
PS["verdict"] = np.where(PS.perm_mean.between(0.46, 0.54), "passes",
                         "INVESTIGATE")
disp(PS.round(4).reset_index(),
     title=f"4a &middot; <b>Permutation control across all {len(ORIGINS)} folds.</b> Shuffled "
           "training labels must produce chance performance on the untouched test month. A single "
           "fold can land anywhere in ±0.07; the <b>mean</b> is the test", save="v10b_permutation")
if HAVE_MPL:
    fig, ax = plt.subplots(figsize=(9.5, 3.8))
    for i, fs in enumerate(FS_ORDER):
        d = PERM[PERM.feature_set == fs]
        ax.scatter(d.auc_perm, np.full(len(d), i), s=55, color=FSCOL[fs],
                   alpha=.85, edgecolor="white", linewidth=.8)
        ax.scatter([d.auc_perm.mean()], [i], marker="|", s=600, color=INK, linewidth=2)
    ax.axvline(0.5, color=INK, ls=":", lw=1.2)
    ax.set_yticks(range(len(FS_ORDER))); ax.set_yticklabels(FS_ORDER)
    ax.set_xlabel("AUC with permuted training labels"); ax.set_xlim(0.30, 0.70)
    _fin(fig, "4b · Every fold, permuted. The bar is the mean",
         "the dotted line is chance; scatter is fold-to-fold noise", save="v10b_perm_plot")
assert PS.perm_mean.between(0.44, 0.56).all(), (
    "permutation control failed across folds — investigate before trusting anything downstream")


## 5 · Why `payment_only` "won" — measuring the size blindness

Three things are measured directly rather than argued:

1. **Does base rate fall with client size?** If large clients churn less, a model that knows size
   will down-weight them.
2. **Is each model's probability correlated with size?** `payment_only` should be near zero;
   everything containing `ld_bl_log_bal` should be negative.
3. **How concentrated is each queue?** Dollars reached per *distinct* client. A queue that reaches
   more money through fewer names is not finding more attrition — it is finding bigger clients.

In [ ]:
# =====================================================================
# 5 · THE SIZE-BLINDNESS DIAGNOSTIC                      [OUTPUT BLOCK 5]
# =====================================================================
SS = S[S.H == PRIMARY_H].copy()
SS["lbal"] = np.log1p(SS.bar_med12.clip(lower=0))
SS["bal_decile"] = pd.qcut(SS.bar_med12.rank(method="first"), 10, labels=False)+1

# (a) does churn fall with size?
BD = (SS.groupby("bal_decile", as_index=False)
      .agg(n=("y", "size"), median_balance=("bar_med12", "median"),
           base_rate=("y", "mean"), dollars_at_risk=("bar_med12", "sum")))
disp(BD.assign(median_balance=BD.median_balance.map(usd),
               dollars_at_risk=BD.dollars_at_risk.map(usd)).round(4),
     title="5a &middot; <b>Large clients churn less.</b> This is the fact a size-aware model "
           "learns and a size-blind one cannot", n=12, save="v10b_base_by_size")
if HAVE_MPL:
    fig, ax = plt.subplots(figsize=(9.5, 3.6))
    ax.bar(BD.bal_decile, BD.base_rate, color=ACC2)
    ax.set_xlabel("balance decile (10 = largest)"); ax.set_ylabel("6-month event rate")
    ax.yaxis.set_major_formatter(FuncFormatter(lambda v, p: f"{v:.1%}"))
    _fin(fig, "5a · Event rate by client size", "the negative slope is what payment_only is "
         "blind to", save="v10b_base_size_plot")

# (b) is each model's probability aware of size?
rows = []
for fs in FS_ORDER:
    d = SS[["lbal", f"p_{fs}", "y"]].dropna()
    rows.append(dict(feature_set=fs,
                     corr_p_vs_logbalance=float(np.corrcoef(d.lbal, d[f"p_{fs}"])[0, 1]),
                     sees_balance=("ld_bl_log_bal" in FSETS[fs] or "ld_bal_live" in FSETS[fs]),
                     auc=float(FD[PRIMARY_DEF][(FD[PRIMARY_DEF].feature_set == fs) &
                                               (FD[PRIMARY_DEF].H == PRIMARY_H)].auc.mean())))
SZ = pd.DataFrame(rows)
disp(SZ.round(4), title="5b &middot; <b>Correlation between the predicted probability and client "
     "size.</b> Negative means the model has learned that big clients churn less. "
     "<code>payment_only</code> should sit near zero — it cannot see size at all",
     save="v10b_size_corr")

# (c) concentration of each queue
def first_alerts(df, score_col, K, alpha=0.0, bar_col="bar_med12"):
    d = df.copy()
    s = pd.to_numeric(d[score_col], errors="coerce")
    if alpha > 0:
        s = s*np.power(np.maximum(pd.to_numeric(d[bar_col], errors="coerce").fillna(0.), 1.), alpha)
    d["_s"] = s
    d["_r"] = d.groupby("m_idx")["_s"].rank(ascending=False, method="first", na_option="bottom")
    fl = d[d._r <= K]
    first = fl.sort_values(["cust_pwr_id", "m_idx"]).drop_duplicates("cust_pwr_id", keep="first")
    return fl, first

rows = []
for fs in FS_ORDER:
    for alpha in [0.0, 0.75, 1.0]:
        fl, first = first_alerts(SS, f"p_{fs}", QUEUE_K, alpha)
        tp = first[first.y == 1]
        rows.append(dict(feature_set=fs, alpha=alpha, alerts=len(fl),
                         conversations=len(first), tp_clients=int(len(tp)),
                         precision=float(len(tp)/max(len(first), 1)),
                         dollars_reached=float(tp.bar_now.sum()),
                         dollars_per_tp_client=float(tp.bar_now.sum()/max(len(tp), 1)),
                         repeat_rate=float(len(fl)/max(len(first), 1))))
CONC = pd.DataFrame(rows)
disp(CONC.assign(dollars_reached=CONC.dollars_reached.map(usd),
                 dollars_per_tp_client=CONC.dollars_per_tp_client.map(usd)).round(4),
     title="5c &middot; <b>Concentration.</b> At &alpha;=0.75, if <code>payment_only</code> "
           "reaches more dollars through <i>fewer</i> clients at a <i>higher</i> repeat rate, it "
           "is finding bigger names, not more attrition", n=20, save="v10b_concentration")
_p = CONC[(CONC.feature_set == "payment_only") & (CONC.alpha == 0.75)].iloc[0]
_a = CONC[(CONC.feature_set == "all_features") & (CONC.alpha == 0.75)].iloc[0]
kv([("payment_only · dollars reached at alpha=0.75", usd(_p.dollars_reached)),
    ("payment_only · departing clients reached", int(_p.tp_clients)),
    ("payment_only · dollars per client reached", usd(_p.dollars_per_tp_client)),
    ("all_features · dollars reached at alpha=0.75", usd(_a.dollars_reached)),
    ("all_features · departing clients reached", int(_a.tp_clients)),
    ("all_features · dollars per client reached", usd(_a.dollars_per_tp_client)),
    ("payment_only AUC", round(float(SZ[SZ.feature_set == 'payment_only'].auc.iloc[0]), 4)),
    ("all_features AUC", round(float(SZ[SZ.feature_set == 'all_features'].auc.iloc[0]), 4)),
    ("reading", "payment_only reaches money through size blindness, not better detection")],
   title="5d &middot; <b>The answer to &lsquo;why does payment_only win&rsquo;</b>",
   save="v10b_why_payment")


## 6 · The corrected comparison — α = 1 with a calibrated probability

At `α = 1` the score is `p × balance`, which **is** expected dollars at risk. Size enters once,
through the probability and the balance jointly, and cannot be double-counted. That makes it the
principled reference and the only α at which the four feature sets are comparable.

Reported three ways:
- **α = 1**, the fair comparison.
- **α\*** chosen per feature set, since v10 hardcoded 0.75 while §9 had found 0.5 best.
- **α = 0**, today's pure-probability queue, as the floor.

In [ ]:
# =====================================================================
# 6 · SAVINGS, CORRECTED                                 [OUTPUT BLOCK 6]
# =====================================================================
def savings_grid(frame, label_tag):
    rows = []
    for fs in FS_ORDER:
        for alpha in ALPHA_GRID:
            for K in CAPACITY:
                fl, first = first_alerts(frame, f"p_{fs}", K, alpha)
                tp = first[first.y == 1]
                reach = float(tp.bar_now.sum()); conv = len(first)
                for ps in P_SAVE_GRID:
                    saved = reach*ps; cost = conv*RM_COST_PER_CALL
                    rows.append(dict(label=label_tag, feature_set=fs, alpha=alpha, k=K,
                                     p_save=ps, conversations=conv, tp_clients=int(len(tp)),
                                     precision=float(len(tp)/max(conv, 1)),
                                     dollars_reached=reach, saved_annualised=saved*ANNUALISE,
                                     rm_cost=cost, net_annualised=(saved-cost)*ANNUALISE,
                                     breakeven_p_save=cost/max(reach, 1)))
    return pd.DataFrame(rows)

SAV = savings_grid(SS, PRIMARY_DEF)
SAV.to_csv(OUT_DIR / "v10b_savings.csv", index=False)

# α sweep — which alpha each set actually wants
ASW = (SAV[(SAV.k == QUEUE_K) & (SAV.p_save == P_SAVE_BASE)]
       .pivot_table(index="feature_set", columns="alpha",
                    values="dollars_reached").reindex(FS_ORDER))
disp(ASW.apply(lambda c: c.map(usd)).reset_index(),
     title=f"6a &middot; <b>Dollars reached by &alpha;</b>, {QUEUE_K:,} alerts a month. v10 "
           "hardcoded &alpha;=0.75 for every set; each set wants a different one",
     save="v10b_alpha_sweep")
lines(ASW.T, "6a · Each feature set wants a different value weight",
      f"{QUEUE_K:,} alerts/month · calibrated probability", ylab="defendable $ reached",
      xlab="α (value weight)", fmt=usd, colors=FSCOL, save="v10b_alpha_plot")
ASTAR = {fs: float(ASW.loc[fs].idxmax()) for fs in FS_ORDER}

# THE FAIR COMPARISON: alpha = 1
H1 = (SAV[(SAV.k == QUEUE_K) & (SAV.alpha == 1.0)]
      .pivot_table(index="feature_set", columns="p_save",
                   values="saved_annualised").reindex(FS_ORDER))
disp(H1.apply(lambda c: c.map(usd)).reset_index(),
     title="6b &middot; <b>&alpha; = 1, the fair comparison.</b> The score is "
           "<code>p &times; balance</code> — expected dollars at risk, with size counted once",
     save="v10b_headline_alpha1")
heat(H1, "6b · Annualised dollars retained — α = 1, calibrated probability",
     f"{QUEUE_K:,} alerts a month · {PRIMARY_DEF} · H={PRIMARY_H}",
     xlab="RM save rate", save="v10b_heat_alpha1")
bar_grouped(H1.T, "RM save rate", FS_ORDER,
            "6c · Feature sets compared at α = 1, where size cannot be double-counted",
            f"annualised retained · {QUEUE_K:,} alerts/month · calibrated probability",
            ylab="annualised $ retained", colors=FSCOL, save="v10b_bar_alpha1")

UP = pd.DataFrame({f"over deposit_only: {fs}": H1.loc[fs]-H1.loc["deposit_only"]
                   for fs in FS_ORDER[1:]})
disp(UP.apply(lambda c: c.map(usd)).reset_index().rename(columns={"index": "p_save"}),
     title="6d &middot; <b>Uplift over a deposit-only queue at &alpha;=1</b> — what the payment "
           "work is worth once the size artefact is removed", save="v10b_uplift_alpha1")

# per-set optimal alpha, for completeness
rows = []
for fs in FS_ORDER:
    r = SAV[(SAV.feature_set == fs) & (SAV.alpha == ASTAR[fs]) & (SAV.k == QUEUE_K)]
    for ps in P_SAVE_GRID:
        rr = r[r.p_save == ps].iloc[0]
        rows.append(dict(feature_set=fs, alpha_star=ASTAR[fs], p_save=ps,
                         saved_annualised=rr.saved_annualised))
BEST = pd.DataFrame(rows).pivot_table(index="feature_set", columns="p_save",
                                      values="saved_annualised").reindex(FS_ORDER)
disp(BEST.apply(lambda c: c.map(usd)).assign(
        alpha_star=[ASTAR[f] for f in FS_ORDER]).reset_index(),
     title="6e &middot; Each set at <b>its own</b> best &alpha;. Read beside 6b — if the ordering "
           "differs, the difference is the size artefact", save="v10b_best_alpha")

_b1 = SAV[(SAV.feature_set == "all_features") & (SAV.k == QUEUE_K) &
          (SAV.alpha == 1.0) & (SAV.p_save == 0.01)].iloc[0]
_d1 = SAV[(SAV.feature_set == "deposit_only") & (SAV.k == QUEUE_K) &
          (SAV.alpha == 1.0) & (SAV.p_save == 0.01)].iloc[0]
_p1 = SAV[(SAV.feature_set == "payment_only") & (SAV.k == QUEUE_K) &
          (SAV.alpha == 1.0) & (SAV.p_save == 0.01)].iloc[0]
kv([("alert volume", f"{QUEUE_K:,} a month"),
    ("save rate", "1% — the most pessimistic case tested"),
    ("all_features · departing clients reached", int(_b1.tp_clients)),
    ("all_features · defendable dollars", usd(_b1.dollars_reached)),
    ("all_features · retained, annualised", usd(_b1.saved_annualised)),
    ("all_features · net, annualised", usd(_b1.net_annualised)),
    ("all_features · break-even save rate", f"{_b1.breakeven_p_save:.3%}"),
    ("payment_only · retained, annualised", usd(_p1.saved_annualised)),
    ("deposit_only · retained, annualised", usd(_d1.saved_annualised)),
    ("uplift of all_features over deposit_only",
     usd(_b1.saved_annualised-_d1.saved_annualised)),
    ("does all_features now beat payment_only?",
     "yes" if _b1.saved_annualised > _p1.saved_annualised else "no — investigate 5b/5c")],
   title="6f &middot; <b>The corrected case at 1%</b>", save="v10b_case_alpha1")


In [ ]:
# =====================================================================
# 7 · MARGINAL RETURN ON THE EFFICIENT FRONTIER          [OUTPUT BLOCK 7]
# =====================================================================
# v10 took successive differences over non-monotone conversation counts and
# produced -$156,678 at k=100. The upper concave envelope of
# (conversations, dollars) is monotone by construction, and its slopes ARE the
# marginal return per extra conversation.
def frontier(d):
    d = d.sort_values("conversations")
    keep = []
    for r in d.itertuples():
        while keep and r.dollars_reached >= keep[-1][1]:
            keep.pop()
        keep.append((r.conversations, r.dollars_reached, r.k))
    hull = []
    for pt in keep:
        while len(hull) >= 2:
            (x0, y0, _), (x1, y1, _) = hull[-2], hull[-1]
            if (y1-y0)*(pt[0]-x1) <= (pt[1]-y1)*(x1-x0): hull.pop()
            else: break
        hull.append(pt)
    out = pd.DataFrame(hull, columns=["conversations", "dollars_reached", "k"])
    out["marginal_per_conversation"] = (out.dollars_reached.diff()/out.conversations.diff())
    return out

rows = []
for fs in FS_ORDER:
    d = SAV[(SAV.feature_set == fs) & (SAV.alpha == 1.0) &
            (SAV.p_save == P_SAVE_BASE)][["k", "conversations", "dollars_reached"]].drop_duplicates()
    f = frontier(d); f["feature_set"] = fs
    f["marginal_retained_at_1pct"] = f.marginal_per_conversation*0.01
    rows.append(f)
FR = pd.concat(rows, ignore_index=True)
disp(FR.assign(dollars_reached=FR.dollars_reached.map(usd),
               marginal_per_conversation=FR.marginal_per_conversation.map(usd),
               marginal_retained_at_1pct=FR.marginal_retained_at_1pct.map(usd)).round(2),
     title=f"7a &middot; <b>Efficient frontier and marginal return.</b> "
           f"<code>marginal_retained_at_1pct</code> is what one more conversation retains at a 1% "
           f"save rate — <b>stop where it falls below {usd(RM_COST_PER_CALL)}</b>",
     n=40, save="v10b_frontier")
if HAVE_MPL:
    fig, ax = plt.subplots(figsize=(9.5, 4.2))
    for fs in FS_ORDER:
        f = FR[FR.feature_set == fs]
        ax.plot(f.conversations, f.marginal_retained_at_1pct, marker="o", ms=5, lw=2,
                color=FSCOL[fs], label=fs)
    ax.axhline(RM_COST_PER_CALL, color=INK, ls=":", lw=1.4)
    ax.text(ax.get_xlim()[1], RM_COST_PER_CALL, f"  {usd(RM_COST_PER_CALL)} cost of a call",
            fontsize=8, va="bottom", ha="right", color=INK)
    ax.set_yscale("log"); ax.set_xlabel("distinct conversations over 7 months")
    ax.set_ylabel("$ retained by one more conversation, at 1% save")
    ax.yaxis.set_major_formatter(FuncFormatter(lambda v, p: usd(v)))
    ax.legend(frameon=False, fontsize=8)
    _fin(fig, "7a · How many calls? Stop where the curve crosses the cost line",
         "slopes of the upper concave envelope — monotone by construction",
         save="v10b_marginal_plot")


In [ ]:
# =====================================================================
# 8 · D_money_move — the gap by client size              [OUTPUT BLOCK 8]
# =====================================================================
# Median gap came out at 2 months, not the 6-8 the dollar-weighted decay curve
# implied. Both can be true: the decay curve is DOLLAR-weighted, the gap median
# is CLIENT-weighted. If large clients move early, D is still the right target
# and should be defined on a value-weighted basis.
if "lab" not in dir(): lab = spark.read.parquet(v6("labels_customer"))
if "BAL" not in dir(): BAL = spark.read.parquet(v9("balance"))
DL = spark.read.parquet(hp("labels_D"))
GAPD = (lab.filter(F.col("q_A_full_exit").isNotNull())
        .join(DL, "cust_pwr_id", "inner")
        .select("cust_pwr_id", F.col("q_A_full_exit").alias("event_m"),
                (F.col("q_A_full_exit")-F.col("q_D_money_move")).alias("gap"))
        .join(BAL.select("cust_pwr_id", "m_idx", "bal_med12"), "cust_pwr_id", "inner")
        .withColumn("rel_m", F.col("m_idx")-F.col("event_m"))
        .filter(F.col("rel_m") == -12)
        .select("cust_pwr_id", "gap", F.col("bal_med12").alias("bar")))
G = collect_pd(GAPD.filter(F.col("bar") > BAL_FLOOR), "A-D gaps with balance")
G["bal_decile"] = pd.qcut(G.bar.rank(method="first"), 10, labels=False)+1
GS = (G.groupby("bal_decile", as_index=False)
      .agg(clients=("gap", "size"), median_balance=("bar", "median"),
           median_gap=("gap", "median"), p75_gap=("gap", lambda s: s.quantile(.75)),
           share_gap_ge6=("gap", lambda s: float((s >= 6).mean())),
           dollars=("bar", "sum")))
disp(GS.assign(median_balance=GS.median_balance.map(usd), dollars=GS.dollars.map(usd)).round(3),
     title="8a &middot; <b>Does the money move earlier for the clients that hold it?</b> "
           "If the median gap rises with balance decile, <code>D_money_move</code> is still the "
           "right target &mdash; it just needs a value-weighted reading",
     n=12, save="v10b_gap_by_size")
_dw = float((G.gap*G.bar).sum()/G.bar.sum())
kv([("client-weighted median gap (what v10 reported)", f"{G.gap.median():.0f} months"),
    ("DOLLAR-weighted mean gap", f"{_dw:.1f} months"),
    ("median gap, balance deciles 1-3", f"{GS[GS.bal_decile<=3].median_gap.median():.0f}"),
    ("median gap, balance deciles 8-10", f"{GS[GS.bal_decile>=8].median_gap.median():.0f}"),
    ("share of deciles 8-10 with a gap of 6+ months",
     f"{GS[GS.bal_decile>=8].share_gap_ge6.mean():.1%}"),
    ("verdict", "D keeps real lead for the clients that matter"
                if GS[GS.bal_decile >= 8].median_gap.median() >
                   GS[GS.bal_decile <= 3].median_gap.median() + 1
                else "D buys little lead even for large clients — reconsider the target")],
   title="8b &middot; Reconciling the 2-month median with the decay curve",
   save="v10b_gap_verdict")
if HAVE_MPL:
    fig, ax = plt.subplots(figsize=(9.5, 3.8))
    ax.bar(GS.bal_decile, GS.median_gap, color=ACC2, label="median gap")
    ax.plot(GS.bal_decile, GS.p75_gap, marker="o", color=ACC, lw=2, label="p75 gap")
    ax.set_xlabel("balance decile (10 = largest)")
    ax.set_ylabel("months the money move precedes the closure")
    ax.legend(frameon=False, fontsize=8)
    _fin(fig, "8a · Lead bought by the money-move target, by client size",
         f"dollar-weighted mean gap {_dw:.1f} months against a client-weighted median of "
         f"{G.gap.median():.0f}", save="v10b_gap_plot")


In [ ]:
# =====================================================================
# 9 · B_bal_exit FOLDED IN + THE LEAD LADDER PER CLIENT  [OUTPUT BLOCK 9]
# =====================================================================
# (a) the pool, and what excluding B costs the business case
att_A = lab.filter(F.col("q_A_full_exit").isNotNull()).select(
    "cust_pwr_id", F.col("q_A_full_exit").alias("event_m"))
att_B = (lab.filter(F.col("q_B_bal_exit").isNotNull() & F.col("q_A_full_exit").isNull())
         .select("cust_pwr_id", F.col("q_B_bal_exit").alias("event_m")))
def pool(df, nm):
    r = (BAL.join(df, "cust_pwr_id", "inner")
         .withColumn("rel_m", F.col("m_idx")-F.col("event_m"))
         .filter(F.col("rel_m") == -12)
         .agg(F.count(F.lit(1)).alias("n"), F.sum("bal_med12").alias("pool"),
              F.expr("percentile_approx(bal_med12, 0.5)").alias("median")).collect()[0])
    return dict(cohort=nm, clients=int(r["n"]), pool=float(r["pool"] or 0),
                mean=float((r["pool"] or 0)/max(r["n"], 1)), median=float(r["median"] or 0))
PB = pd.DataFrame([pool(att_A, "A_full_exit"), pool(att_B, "B_bal_exit only")])
PB.loc[len(PB)] = dict(cohort="combined", clients=PB.clients.sum(), pool=PB.pool.sum(),
                       mean=PB.pool.sum()/PB.clients.sum(), median=np.nan)
disp(PB.assign(pool=PB.pool.map(usd), mean=PB.mean.map(usd), median=PB["median"].map(usd)),
     title="9a &middot; <b>The pool, with B included.</b> B clients drain without closing, are "
           "invisible to balance monitoring, and hold far more each", save="v10b_pool")
kv([("A pool", usd(PB.iloc[0]["pool"])), ("B-only pool", usd(PB.iloc[1]["pool"])),
    ("combined", usd(PB.iloc[2]["pool"])),
    ("mean relationship, A", usd(PB.iloc[0]["mean"])),
    ("mean relationship, B-only", usd(PB.iloc[1]["mean"])),
    ("understatement from excluding B",
     f"{PB.iloc[1]['pool']/max(PB.iloc[0]['pool'],1):.0%} of the A pool")],
   title="9b &middot; What v10 left out", save="v10b_pool_verdict")

# (b) savings on the UNION label
SAB = SC["AB_any"]; SAB = SAB[SAB.H == PRIMARY_H]
SAV_AB = savings_grid(SAB, "AB_any")
CMP = pd.concat([
    SAV[(SAV.alpha == 1.0) & (SAV.k == QUEUE_K) & (SAV.p_save == 0.01)]
        .assign(label="A only")[["label", "feature_set", "tp_clients",
                                 "dollars_reached", "saved_annualised"]],
    SAV_AB[(SAV_AB.alpha == 1.0) & (SAV_AB.k == QUEUE_K) & (SAV_AB.p_save == 0.01)]
        .assign(label="A or B")[["label", "feature_set", "tp_clients",
                                 "dollars_reached", "saved_annualised"]]])
piv = CMP.pivot_table(index="feature_set", columns="label",
                      values="saved_annualised").reindex(FS_ORDER)
disp(piv.apply(lambda c: c.map(usd)).reset_index(),
     title="9c &middot; <b>Annualised retained at a 1% save rate, A only against A-or-B.</b> "
           "Counting the drained-but-open clients is a pure addition — nobody was being alerted "
           "on them before", save="v10b_savings_AB")
bar_grouped(piv, "feature set", list(piv.columns),
            "9c · Including the drained-but-open population", 
            f"α=1 · {QUEUE_K:,} alerts/month · 1% save rate · annualised",
            ylab="annualised $ retained", save="v10b_bar_AB")

# (c) the lead ladder, normalised per client
fl, first = first_alerts(SS, "p_all_features", QUEUE_K, 1.0)
tp = first[first.y == 1][["cust_pwr_id", "m_idx", "event_m", "bar_now"]]
caught = spark.createDataFrame([(str(a), int(b)) for a, b in zip(tp.cust_pwr_id, tp.event_m)],
                               ["cust_pwr_id", "event_m"])
BB = (BAL.select("cust_pwr_id", "m_idx", "bal_now")
      .join(F.broadcast(caught), "cust_pwr_id", "inner")
      .withColumn("rel_m", F.col("m_idx")-F.col("event_m")))
LD = collect_pd(BB.filter(F.col("rel_m").isin([-l for l in LEAD_GRID]))
                .groupBy("rel_m").agg(F.sum("bal_now").alias("dollars"),
                                      F.countDistinct("cust_pwr_id").alias("clients")),
                "lead ladder")
LD["lead_months"] = -LD.rel_m
LD["per_client"] = LD.dollars/LD.clients
LD = LD.sort_values("lead_months")
disp(LD.assign(dollars=LD.dollars.map(usd), per_client=LD.per_client.map(usd))
     [["lead_months", "clients", "dollars", "per_client"]],
     title="9d &middot; <b>Lead ladder, normalised per client.</b> v10's 12-month rung was lower "
           "than the 9-month rung only because fewer clients have 12 months of history &mdash; "
           "<code>per_client</code> is the comparable column", save="v10b_lead_per_client")
if HAVE_MPL:
    fig, ax1 = plt.subplots(figsize=(9.5, 4.0))
    ax1.bar(LD.lead_months, LD.per_client, color=ACC2, alpha=.85, label="per client")
    ax1.set_xlabel("months of lead"); ax1.set_ylabel("defendable $ per client")
    ax1.yaxis.set_major_formatter(FuncFormatter(lambda v, p: usd(v)))
    ax2 = ax1.twinx(); ax2.plot(LD.lead_months, LD.clients, marker="o", color=ACC, lw=2,
                                label="clients with that much history")
    ax2.set_ylabel("clients"); ax2.grid(False)
    _fin(fig, "9d · Earlier detection, per client — the comparable view",
         "the raw sum falls at 12 months because the denominator shrinks",
         save="v10b_lead_per_client_plot")
_pc = LD.set_index("lead_months").per_client
kv([("defendable per client at 2 months' lead", usd(_pc.get(2, np.nan))),
    ("...at 4", usd(_pc.get(4, np.nan))), ("...at 6", usd(_pc.get(6, np.nan))),
    ("...at 9", usd(_pc.get(9, np.nan))), ("...at 12", usd(_pc.get(12, np.nan))),
    ("multiple, 2 -> 6 months", f"{_pc.get(6, np.nan)/max(_pc.get(2, np.nan), 1):.1f}x"),
    ("multiple, 2 -> 9 months", f"{_pc.get(9, np.nan)/max(_pc.get(2, np.nan), 1):.1f}x")],
   title="9e &middot; <b>What earlier detection is worth per client</b> — the number that should "
         "set the v11 agenda", save="v10b_lead_verdict")


---

## What this run should settle

1. **Every expected-dollar figure in v10 was inflated ~2.2×.** The ordering was unaffected, so no
   conclusion about *which* feature set wins changes — but nothing from v10 §7 or §8 should be
   quoted in dollars. Use 6b.

2. **If §6b puts `all_features` ahead of `payment_only` at α=1**, the v10 result is confirmed as a
   size artefact and the corrected uplift in 6d is the number that answers "was the payment work
   worth doing". If `payment_only` still wins at α=1 with a calibrated probability, that is a real
   and surprising finding and it deserves a separate investigation — start with 5b, because it
   would mean the balance features are actively hurting the ranking.

3. **§8 decides the fate of `D_money_move`.** A rising gap across balance deciles means the target
   is right and only its *definition* needs to be value-weighted. A flat gap means the money-move
   label buys little lead even for the clients that matter, and the v11 agenda should be the
   longer horizon instead.

4. **§9a is a straightforward addition to the business case.** The B-only pool is a population
   nobody is alerting on today, and it is the one where balance monitoring is provably useless
   (12% precision retention in v7). Every dollar there is incremental.

5. **§9e should set the v11 agenda.** If defendable balance per client at 6 months' lead is several
   times the figure at 2 months, then horizon beats features and the next release is about
   predicting earlier, not seeing more.

## Still outstanding, unchanged
No live evidence that an alert changes an outcome; the 20% of the deposit book invisible to
payments has never been profiled; and `p_save` needs a pilot with a **holdout arm**, because in
this data a save and a client who was never going to leave are indistinguishable.
